# Step 5 — LLM-as-a-Judge

This notebook builds an automated LLM judge that grades every product description with the same rubric used for human rating (`docs/evaluation_rubric.md`).

**Judge model:** `google/gemma-2-9b-it-fast`
**Criteria evaluated:** Fluency, Grammar, Tone, Length, Grounding. Cost and Latency are measured programmatically.
**Output format:** structured JSON, enforced with a Pydantic schema and the API's structured-output mode
**Results:** `results/step5_judge_scores.xlsx`

## 0. Install dependencies

In [2]:
#!pip install openai pandas openpyxl python-dotenv pydantic

## 1. Load API key

In [4]:
from dotenv import load_dotenv
load_dotenv()
print("API key loaded.")

API key loaded.


## 2. Imports & client setup

In [41]:
import os
import json
import time
import pandas as pd
from enum import Enum
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)

JUDGE_MODEL = "google/gemma-2-9b-it-fast"

SPREADSHEET_PATH = "../results/step3_human_ratings.xlsx"

print(f"Judge model: {JUDGE_MODEL}")

Judge model: google/gemma-2-9b-it-fast


## 3. Pydantic output schema

### Why `explanation` comes before `verdict` in the schema

LLMs generate tokens sequentially — each token is conditioned on everything that came before it.
By placing `explanation` first in the schema, the model is forced to **reason out loud before committing to a verdict**.
This mirrors chain-of-thought prompting: the model cannot "decide" and then reverse-engineer a justification —
it must build its argument first, and the verdict emerges naturally from that argument.

If `verdict` came first, the model would pick a label and then generate a post-hoc explanation
that rationalises the pre-committed answer. This leads to less consistent and less accurate verdicts,
especially on borderline cases where the correct label depends on careful reasoning.

In structured output mode, the JSON is generated key-by-key in schema order,
so field ordering is a direct lever on model reasoning quality.

In [46]:
class Verdict(str, Enum):
    good = "good"
    ok   = "ok"
    bad  = "bad"


class CriterionScore(BaseModel):
    # IMPORTANT: explanation MUST come before verdict.
    # In structured output, JSON keys are generated in schema-definition order.
    # Generating the explanation first forces the model to reason through evidence
    # before committing to a label — like thinking before answering.
    # If verdict came first, the model would pick a label and reverse-engineer
    # a justification, producing less reliable scores on borderline cases.
    explanation: str   # reasoning for the verdict — generated FIRST
    verdict: Verdict   # the actual rating — generated AFTER the reasoning


class DescriptionEvaluation(BaseModel):
    """Full evaluation of one product description across all 5 per-description criteria."""
    fluency:   CriterionScore
    grammar:   CriterionScore
    tone:      CriterionScore
    length:    CriterionScore
    grounding: CriterionScore


print("Schema defined.")
print(json.dumps(DescriptionEvaluation.model_json_schema(), indent=2))

Schema defined.
{
  "$defs": {
    "CriterionScore": {
      "properties": {
        "explanation": {
          "title": "Explanation",
          "type": "string"
        },
        "verdict": {
          "$ref": "#/$defs/Verdict"
        }
      },
      "required": [
        "explanation",
        "verdict"
      ],
      "title": "CriterionScore",
      "type": "object"
    },
    "Verdict": {
      "enum": [
        "good",
        "ok",
        "bad"
      ],
      "title": "Verdict",
      "type": "string"
    }
  },
  "description": "Full evaluation of one product description across all 5 per-description criteria.",
  "properties": {
    "fluency": {
      "$ref": "#/$defs/CriterionScore"
    },
    "grammar": {
      "$ref": "#/$defs/CriterionScore"
    },
    "tone": {
      "$ref": "#/$defs/CriterionScore"
    },
    "length": {
      "$ref": "#/$defs/CriterionScore"
    },
    "grounding": {
      "$ref": "#/$defs/CriterionScore"
    }
  },
  "required": [
    "fluency",
   

## 4. Judge system prompt

The judge prompt embeds the full Step 1 rubric definitions verbatim so the model
applies identical standards to those used during manual evaluation.

Grounding requires special attention: the judge must receive both the **original product data**
and the **generated description** to check whether every claim in the description
is traceable to the input. Without the original data, the judge cannot detect fabrications.

In [94]:
JUDGE_SYSTEM_PROMPT = """You are a strict e-commerce content quality judge. Evaluate the generated product description against the original product data using this rubric. Return structured JSON only.

IMPORTANT: Keep each explanation under 20 words. Be concise.

RUBRIC:
1. FLUENCY: good=natural flowing prose, no awkward phrasing; ok=1-2 awkward phrases but readable; bad=multiple unnatural constructions or repetitive phrasing.
2. GRAMMAR: good=zero errors; ok=1-2 minor errors that don't distort meaning; bad=3+ errors or meaning-distorting errors.
3. TONE: good=warm, confident, persuasive like a trusted retailer; ok=slightly flat or slightly over-hyped but acceptable; bad=purely clinical/robotic or aggressively spammy.
4. LENGTH: good=50-90 words; ok=40-49 or 91-110 words; bad=under 40 or over 110 words. Count the words carefully.
5. GROUNDING: good=every specific claim traceable to input data; ok=1 minor reasonable inference, no fabrications; bad=at least one fabricated claim not in the input. WARNING: battery/power claims for non-electronic products (LEGO sets, SSDs, storage devices) are fabrications and must be rated bad.

For each criterion write explanation first (max 20 words), then verdict (good/ok/bad)."""

## 5. Judge function

In [96]:
def build_judge_user_message(row: pd.Series) -> str:
    """
    Build the user message for the judge.
    Includes original product data AND the generated description.
    Grounding evaluation requires both to detect fabrications.
    """
    return (
        f"=== ORIGINAL PRODUCT DATA ===\n"
        f"Product name: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}\n\n"
        f"=== GENERATED DESCRIPTION TO EVALUATE ===\n"
        f"{row['generated_description']}"
    )


def judge_description(row: pd.Series, model: str = JUDGE_MODEL) -> dict:
    """
    Call the judge model for a single product row using structured output.

    Returns a dict with per-criterion explanation+verdict pairs,
    plus latency_ms, input_tokens, output_tokens.
    """
    user_message = build_judge_user_message(row)

    start = time.perf_counter()
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user",   "content": [{"type": "text", "text": user_message}]}
        ],
        response_format=DescriptionEvaluation,
        temperature=0.1,
        max_tokens=2500,   
    )
    latency_ms = (time.perf_counter() - start) * 1000

    evaluation: DescriptionEvaluation = response.choices[0].message.parsed

    return {
        "judge_fluency_explanation":   evaluation.fluency.explanation,
        "judge_fluency":               evaluation.fluency.verdict.value,
        "judge_grammar_explanation":   evaluation.grammar.explanation,
        "judge_grammar":               evaluation.grammar.verdict.value,
        "judge_tone_explanation":      evaluation.tone.explanation,
        "judge_tone":                  evaluation.tone.verdict.value,
        "judge_length_explanation":    evaluation.length.explanation,
        "judge_length":                evaluation.length.verdict.value,
        "judge_grounding_explanation": evaluation.grounding.explanation,
        "judge_grounding":             evaluation.grounding.verdict.value,
        "judge_latency_ms":            round(latency_ms, 1),
        "judge_input_tokens":          response.usage.prompt_tokens,
        "judge_output_tokens":         response.usage.completion_tokens,
    }

print("Judge function defined.")

Judge function defined.


## 6. Pass / fail logic (same rules as Step 1)

In [55]:
def compute_judge_final_score(row: pd.Series) -> str:
    """
    Apply Step 1 pass/fail rules to judge verdicts.

    Go / no-go rules (automatic fail):
      - grounding == bad
      - length == bad

    Cumulative pass bar:
      - zero bad ratings across all 5 criteria
      - at least 3 good ratings
    """
    criteria = ['judge_fluency', 'judge_grammar', 'judge_tone',
                'judge_length', 'judge_grounding']
    scores = [row[c] for c in criteria]

    # Go / no-go rules
    if row['judge_grounding'] == 'bad': return 'fail'
    if row['judge_length']    == 'bad': return 'fail'

    # Cumulative bar
    if scores.count('bad')  > 0: return 'fail'
    if scores.count('good') < 3: return 'fail'

    return 'pass'

print("Pass/fail logic defined.")

Pass/fail logic defined.


## 7. Load the spreadsheet

In [58]:
df = pd.read_excel(SPREADSHEET_PATH)
print(f"Loaded {len(df)} rows.")
print(f"Columns: {list(df.columns)}")

# Add blank judge columns
judge_cols = [
    'judge_fluency_explanation',   'judge_fluency',
    'judge_grammar_explanation',   'judge_grammar',
    'judge_tone_explanation',      'judge_tone',
    'judge_length_explanation',    'judge_length',
    'judge_grounding_explanation', 'judge_grounding',
    'judge_final_score',
    'judge_latency_ms', 'judge_input_tokens', 'judge_output_tokens',
]
for col in judge_cols:
    if col not in df.columns:
        df[col] = None

df.head(2)

Loaded 50 rows.
Columns: ['product_name', 'Product_attribute_list', 'material', 'warranty', 'generated_description', 'latency_ms', 'input_tokens', 'output_tokens', 'fluency', 'grammar', 'tone', 'length', 'grounding', 'latency_rating', 'cost_rating', 'cost_usd', 'final_score', 'Unnamed: 17', 'In', 0.03, 'Out', 0.09]


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,...,judge_tone_explanation,judge_tone,judge_length_explanation,judge_length,judge_grounding_explanation,judge_grounding,judge_final_score,judge_latency_ms,judge_input_tokens,judge_output_tokens
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Built for everyday reliability, the Apple iPho...",864.7,359,89,good,good,...,None,None,None,None,None,None,None,None,None,None
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture life's moments with the Samsung Galaxy...,1008.0,362,85,good,good,...,None,None,None,None,None,None,None,None,None,None


## 8. Sanity check — run on 5 products first

Before the full batch, the judge runs on 5 hand-picked products and we review the explanations manually:
- Are the explanations specific to the product, or generic?
- Does the judge count words correctly for Length?
- Does the judge detect fabricated claims for Grounding?
- Are the verdicts consistent with what a human would give?

In [61]:
# Sanity check on 5 products — indices chosen to include a variety:
# row 0 (iPhone — expected all good), row 3 (Sony headphones),
# row 11 (GoPro), row 30 (LEGO — previous grounding fail), row 43 (SSD — previous grounding fail)
SANITY_INDICES = [0, 3, 11, 30, 43]

sanity_results = []
for idx in SANITY_INDICES:
    row = df.iloc[idx]
    print(f"\n{'='*60}")
    print(f"[{idx:02d}] {row['product_name']}")
    print(f"Description ({len(str(row['generated_description']).split())} words):")
    print(row['generated_description'])
    print()
    try:
        result = judge_description(row)
        sanity_results.append((idx, result))
        for crit in ['fluency','grammar','tone','length','grounding']:
            print(f"  {crit.upper()}: {result[f'judge_{crit}'].upper()}")
            print(f"    → {result[f'judge_{crit}_explanation']}")
        print(f"  Latency: {result['judge_latency_ms']:.0f} ms")
    except Exception as e:
        print(f"  ERROR: {e}")


[00] Apple iPhone 15 Pro
Description (68 words):
Built for everyday reliability, the Apple iPhone 15 Pro combines a durable titanium frame with Ceramic Shield glass to deliver lasting value. With its compact design, it's perfect for on-the-go use. The A17 Pro chip and 120 Hz ProMotion display ensure seamless performance and stunning visuals. Fast charging via USB-C keeps you powered up. Backed by a 1-year limited warranty, it's a purchase you can make with confidence.

  ERROR: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=800, prompt_tokens=1783, total_tokens=2583, completion_tokens_details=None, prompt_tokens_details=None)

[03] Sony WH‑1000XM5 Headphones
Description (62 words):
Immerse yourself in exceptional sound with the Sony WH‑1000XM5 Headphones. Featuring active noise cancelling technology, these headphones deliver a distraction-free listening experience. Enjoy up to 30 hours of battery life and seamless connectivity with

### Sanity check review

Questions checked before the full run:

1. **Length:** does the judge count words correctly? Row 0 has about 68 words, so it should be `good`.
2. **Grounding:** for row 30 (LEGO) and row 43 (SSD), does the judge catch the battery hallucination in the baseline descriptions?
3. **Explanation quality:** are the explanations specific to the product, or boilerplate?
4. **Consistency:** would a human agree with the verdicts?

Findings are covered in Step 6. The judge **miscounts words**, so Length is overridden with a programmatic count there.

## 9. Full run — judge all 50 products

In [98]:
# Only run after sanity check passes
errors = []

for idx, row in df.iterrows():
    print(f"[{idx+1:02d}/50] {row['product_name'][:45]} ... ", end="")

    if pd.isna(row['generated_description']):
        print("SKIP (no description)")
        continue

    try:
        result = judge_description(row)

        for col, val in result.items():
            df.at[idx, col] = val

        # Compute final score immediately
        df.at[idx, 'judge_final_score'] = compute_judge_final_score(df.iloc[idx])

        print(
            f"OK  "
            f"F={result['judge_fluency'][0].upper()} "
            f"G={result['judge_grammar'][0].upper()} "
            f"T={result['judge_tone'][0].upper()} "
            f"L={result['judge_length'][0].upper()} "
            f"Gr={result['judge_grounding'][0].upper()} "
            f"→ {df.at[idx,'judge_final_score'].upper()} "
            f"({result['judge_latency_ms']:.0f}ms)"
        )

    except Exception as e:
        print(f"ERROR: {e}")
        errors.append((idx, str(e)))

print(f"\nDone. Errors: {len(errors)}")
if errors:
    print("Failed rows:", errors)

[01/50] Apple iPhone 15 Pro ... OK  F=G G=G T=G L=G Gr=G → PASS (5911ms)
[02/50] Samsung Galaxy S24 Ultra ... OK  F=G G=G T=G L=G Gr=G → PASS (1424ms)
[03/50] Google Pixel 8 Pro ... OK  F=G G=G T=G L=G Gr=G → PASS (1358ms)
[04/50] Sony WH‑1000XM5 Headphones ... OK  F=G G=G T=G L=G Gr=G → PASS (1554ms)
[05/50] Bose QuietComfort Ultra Earbuds ... OK  F=G G=G T=G L=G Gr=G → PASS (1490ms)
[06/50] Amazon Echo Dot (5th Gen) ... OK  F=G G=G T=G L=G Gr=G → PASS (1530ms)
[07/50] Dell XPS 13 9310 Laptop ... OK  F=G G=G T=G L=G Gr=G → PASS (1638ms)
[08/50] Apple MacBook Air 13″ (M3) ... OK  F=G G=G T=G L=G Gr=G → PASS (1464ms)
[09/50] Microsoft Surface Pro 10 ... OK  F=G G=G T=G L=G Gr=G → PASS (1918ms)
[10/50] Garmin Forerunner 255 ... OK  F=G G=G T=G L=G Gr=G → PASS (1527ms)
[11/50] Fitbit Charge 6 ... OK  F=G G=G T=G L=G Gr=G → PASS (1392ms)
[12/50] GoPro HERO12 Black ... OK  F=G G=G T=G L=O Gr=G → PASS (1508ms)
[13/50] DJI Mini 4 Pro Drone ... OK  F=G G=G T=G L=G Gr=G → PASS (1557ms)
[14/50] 

In [102]:
# Switch to Qwen for the stubborn row 39
idx = 39
row = df.iloc[idx]
print(f"Retrying [{idx}] {row['product_name']} with Qwen...")

try:
    start = time.perf_counter()
    response = client.beta.chat.completions.parse(
        model="Qwen/Qwen3-30B-A3B-Instruct-2507",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user",   "content": [{"type": "text", "text": build_judge_user_message(row)}]}
        ],
        response_format=DescriptionEvaluation,
        temperature=0.1,
        max_tokens=2000,
    )
    latency_ms = (time.perf_counter() - start) * 1000
    evaluation = response.choices[0].message.parsed

    df.at[idx, 'judge_fluency_explanation']   = evaluation.fluency.explanation
    df.at[idx, 'judge_fluency']               = evaluation.fluency.verdict.value
    df.at[idx, 'judge_grammar_explanation']   = evaluation.grammar.explanation
    df.at[idx, 'judge_grammar']               = evaluation.grammar.verdict.value
    df.at[idx, 'judge_tone_explanation']      = evaluation.tone.explanation
    df.at[idx, 'judge_tone']                  = evaluation.tone.verdict.value
    df.at[idx, 'judge_length_explanation']    = evaluation.length.explanation
    df.at[idx, 'judge_length']                = evaluation.length.verdict.value
    df.at[idx, 'judge_grounding_explanation'] = evaluation.grounding.explanation
    df.at[idx, 'judge_grounding']             = evaluation.grounding.verdict.value
    df.at[idx, 'judge_latency_ms']            = round(latency_ms, 1)
    df.at[idx, 'judge_input_tokens']          = response.usage.prompt_tokens
    df.at[idx, 'judge_output_tokens']         = response.usage.completion_tokens
    df.at[idx, 'judge_final_score']           = compute_judge_final_score(df.iloc[idx])

    print(f"OK → {df.at[idx, 'judge_final_score'].upper()}")

except Exception as e:
    print(f"ERROR: {e}")

Retrying [39] Blink Outdoor 4 with Qwen...
OK → PASS


## 10. Summary statistics

In [104]:
judged = df[df['judge_final_score'].notna()]

print(f"=== JUDGE RESULTS ({len(judged)} products) ===")
print(f"Pass: {(judged['judge_final_score']=='pass').sum()}")
print(f"Fail: {(judged['judge_final_score']=='fail').sum()}")
print()

for crit in ['judge_fluency','judge_grammar','judge_tone','judge_length','judge_grounding']:
    vc = judged[crit].value_counts()
    label = crit.replace('judge_', '')
    print(f"{label:12s}: good={vc.get('good',0):2d}  ok={vc.get('ok',0):2d}  bad={vc.get('bad',0):2d}")

print()
print(f"Avg judge latency: {judged['judge_latency_ms'].mean():.0f} ms")
print(f"Avg judge input tokens:  {judged['judge_input_tokens'].mean():.0f}")
print(f"Avg judge output tokens: {judged['judge_output_tokens'].mean():.0f}")

# Judge cost (same pricing as generation model: $0.03/$0.09 per 1M)
INPUT_PRICE  = 0.03 / 1_000_000
OUTPUT_PRICE = 0.09 / 1_000_000
avg_judge_cost = (
    judged['judge_input_tokens'].mean()  * INPUT_PRICE +
    judged['judge_output_tokens'].mean() * OUTPUT_PRICE
)
print(f"Avg judge cost per call: ${avg_judge_cost:.6f}")

=== JUDGE RESULTS (50 products) ===
Pass: 47
Fail: 3

fluency     : good=47  ok= 3  bad= 0
grammar     : good=49  ok= 1  bad= 0
tone        : good=49  ok= 1  bad= 0
length      : good=37  ok=12  bad= 1
grounding   : good=45  ok= 3  bad= 2

Avg judge latency: 1613 ms
Avg judge input tokens:  777
Avg judge output tokens: 180
Avg judge cost per call: $0.000040


## 11. Agreement with manual scores (Step 3 rated rows)

Compare judge verdicts against human verdicts for the 15 manually rated products.

In [106]:
# Step 3 manually rated row indices
df = df.loc[:, [c for c in df.columns if isinstance(c, str) and not c.startswith('Unnamed')]]

RATED_INDICES = [0,1,2,3,5,8,11,16,19,21,27,30,36,43,49]

rated = df.iloc[RATED_INDICES].copy()

# Check what human score columns actually exist
print("Available columns:", [c for c in df.columns if isinstance(c, str) and not c.startswith('judge_')])
print()

# Human score column names — adjust if yours differ
HUMAN_COL_MAP = {
    'fluency':   'fluency',
    'grammar':   'grammar', 
    'tone':      'tone',
    'length':    'length',
    'grounding': 'grounding',
}

print("=== AGREEMENT: Human vs Judge (15 rated products) ===")
print()

for crit, human_col in HUMAN_COL_MAP.items():
    judge_col = f'judge_{crit}'

    if human_col not in df.columns:
        print(f"{crit:12s}: SKIP — human column '{human_col}' not found in spreadsheet")
        continue
    if judge_col not in df.columns:
        print(f"{crit:12s}: SKIP — judge column '{judge_col}' not found")
        continue

    human_scores = rated[human_col].values
    judge_scores = rated[judge_col].values

    pairs = [(h, j) for h, j in zip(human_scores, judge_scores)
             if h is not None and j is not None
             and not (isinstance(h, float) and pd.isna(h))
             and not (isinstance(j, float) and pd.isna(j))]

    agree = sum(h == j for h, j in pairs)
    total = len(pairs)
    pct = (agree / total * 100) if total > 0 else 0
    print(f"{crit:12s}: agreement {agree}/{total} ({pct:.0f}%)")

    # Show disagreements
    for i, (idx, row) in enumerate(rated.iterrows()):
        h = row.get(human_col)
        j = row.get(judge_col)
        if (h and j 
            and not (isinstance(h, float) and pd.isna(h))
            and not (isinstance(j, float) and pd.isna(j))
            and h != j):
            print(f"  Disagree [{idx:02d}] {str(row['product_name'])[:35]:35s}  human={h}  judge={j}")

Available columns: ['product_name', 'Product_attribute_list', 'material', 'warranty', 'generated_description', 'latency_ms', 'input_tokens', 'output_tokens', 'fluency', 'grammar', 'tone', 'length', 'grounding', 'latency_rating', 'cost_rating', 'cost_usd', 'final_score', 'In', 'Out']

=== AGREEMENT: Human vs Judge (15 rated products) ===

fluency     : agreement 12/15 (80%)
  Disagree [02] Google Pixel 8 Pro                   human=ok  judge=good
  Disagree [08] Microsoft Surface Pro 10             human=ok  judge=good
  Disagree [43] Samsung 980 Pro 2 TB NVMe SSD        human=ok  judge=good
grammar     : agreement 14/15 (93%)
  Disagree [30] LEGO Star Wars Millennium Falcon 75  human=good  judge=ok
tone        : agreement 11/15 (73%)
  Disagree [02] Google Pixel 8 Pro                   human=ok  judge=good
  Disagree [08] Microsoft Surface Pro 10             human=ok  judge=good
  Disagree [19] Dyson V15 Detect Vacuum              human=ok  judge=good
  Disagree [43] Samsung 980 Pro 2 

## 12. Save results back to spreadsheet

In [110]:
OUTPUT_PATH = "../results/step5_judge_scores.xlsx"

df.to_excel(OUTPUT_PATH, index=False, engine='openpyxl')
print(f"Saved {len(df)} rows to '{OUTPUT_PATH}'")
print(f"New columns added: {[c for c in df.columns if c.startswith('judge_')]}")

Saved 50 rows to '../results/step5_judge_scores.xlsx'
New columns added: ['judge_fluency_explanation', 'judge_fluency', 'judge_grammar_explanation', 'judge_grammar', 'judge_tone_explanation', 'judge_tone', 'judge_length_explanation', 'judge_length', 'judge_grounding_explanation', 'judge_grounding', 'judge_final_score', 'judge_latency_ms', 'judge_input_tokens', 'judge_output_tokens']


## 13. Note on model choice

`google/gemma-2-9b-it-fast` is the default judge: small, fast and cheap. For one row where it failed to return a valid structured output, the call falls back to `Qwen/Qwen3-30B-A3B-Instruct-2507` (section 9).

The smaller judge has limits in detecting subtle grounding violations and producing specific explanations. These are analysed in Step 6.